In [2]:
from tcrtrifold.utils import FORMAT_ANTIGEN_COLS
import polars as pl

assay_type = pl.read_parquet("../../data/iedb_meta/assay_type.parquet")
receptor_reference = pl.read_parquet("../../data/iedb_meta/receptor_reference.parquet")


iedb_II_annot = pl.read_parquet(
    "../../data/iedb_II_full/triad/iedb_II_triad.annotated.parquet"
)

iedb_II_pdb_pmhc = (
    iedb_II_annot.filter(pl.col("pmhc_in_pdb")).select(FORMAT_ANTIGEN_COLS).unique()
)

iedb_II_pdb_triad = (
    iedb_II_annot.explode("receptor_id")
    .explode("references")
    .join(
        receptor_reference,
        left_on=["references", "receptor_id"],
        right_on=["reference_id", "receptor_id"],
    )
)

iedb_II = pl.read_parquet(
    "../../data/iedb_II/triad/iedb_II_triad.conf_af3.parquet"
).join(
    iedb_II_pdb_triad.select("job_name", "assay_type", "triad_in_pdb"), on="job_name"
)

iedb_I_annot = pl.read_parquet(
    "../../data/iedb_I_full/triad/iedb_I_triad.annotated.parquet"
)

iedb_I_pdb_pmhc = (
    iedb_II_annot.filter(pl.col("pmhc_in_pdb")).select(FORMAT_ANTIGEN_COLS).unique()
)

iedb_I_pdb_triad = (
    iedb_I_annot.explode("receptor_id")
    .explode("references")
    .join(
        receptor_reference,
        left_on=["references", "receptor_id"],
        right_on=["reference_id", "receptor_id"],
    )
)

iedb_I = pl.read_parquet(
    "../../data/iedb_I/triad/iedb_I_triad.conf_af3.parquet"
).join(
    iedb_I_pdb_triad.select("job_name", "assay_type", "triad_in_pdb"), on="job_name"
)

## Summary nums

In [9]:
iedb_I.filter(pl.col("triad_in_pdb")).select("job_name").unique().height

393

In [10]:
iedb_II.filter(pl.col("triad_in_pdb")).select("job_name").unique().height

63

In [6]:
iedb_I.filter(~pl.col("triad_in_pdb")).filter(pl.col("assay_type") == "x-ray crystallography").select("job_name").item()

'121f50b609621ebe92777820dad90eb9'

In [36]:
iedb_II.filter(
    pl.col("assay_type") == "x-ray crystallography"
).filter(~pl.col("triad_in_pdb")).select("mean_p_tcr_pae").to_series().to_numpy()

iedb_II.filter(
    pl.col("assay_type") == "x-ray crystallography"
).filter(pl.col("triad_in_pdb")).select("mean_p_tcr_pae").to_series().to_numpy()

array([18.04374123,  5.43211937, 11.87347412,  5.33814955,  5.11006117,
        4.50323057,  7.79586029,  7.50704002, 10.60125923,  5.37580204,
       11.39432812, 22.31526184,  5.66939402, 23.68894958,  4.81122828,
       21.07127571,  5.32468271, 10.86191559, 14.77916336, 13.48510456,
        9.68382168,  5.36084175, 10.53259087, 16.21871376, 16.21871376,
       16.21871376,  9.67007446, 18.21819878,  8.86339378,  5.08924675,
        4.33785868,  5.99226379,  7.38107443,  7.65907764,  5.32265091,
        5.30932617])

job_name,cognate,peptide,mhc_class,mhc_1_chain,mhc_1_species,mhc_1_name,mhc_1_seq,mhc_2_chain,mhc_2_species,mhc_2_name,mhc_2_seq,tcr_1_chain,tcr_1_species,tcr_1_seq,tcr_2_chain,tcr_2_species,tcr_2_seq,tcr_1_cdr_1,tcr_1_cdr_2,tcr_1_cdr_2_5,tcr_1_cdr_3,tcr_2_cdr_1,tcr_2_cdr_2,tcr_2_cdr_2_5,tcr_2_cdr_3,receptor_id,references,pmhc_in_validation,chain_iptm,chain_pair_iptm,chain_pair_pae_min,chain_ptm,fraction_disordered,has_clash,iptm,ptm,…,mean_tcr_pmhc_interface_contact_prob,mean_pmhc_tcr_interface_contact_prob,mean_p_tcr_pae,mean_tcr_p_pae,mean_mhc_tcr_pae,mean_tcr_mhc_pae,mean_p_tcr_contact_prob,mean_tcr_p_contact_prob,mean_mhc_tcr_contact_prob,mean_tcr_mhc_contact_prob,tcr_mhc_contacts,tcr_p_contacts,contact_map,peptide_mean_pLDDT,tcr_1_cdr_1_mean_pLDDT,tcr_1_cdr_2_mean_pLDDT,tcr_1_cdr_2_5_mean_pLDDT,tcr_1_cdr_3_mean_pLDDT,tcr_2_cdr_1_mean_pLDDT,tcr_2_cdr_2_mean_pLDDT,tcr_2_cdr_2_5_mean_pLDDT,tcr_2_cdr_3_mean_pLDDT,tcr_cdrs_mean_pLDDT,mhc_helices_mean_pLDDT,mean_p_mhc_pae,mean_mhc_p_pae,mean_mhc_p_interface_pae,mean_p_mhc_interface_pae,mean_mhc_p_interface_contact_prob,mean_p_mhc_interface_contact_prob,min_p_tcr_pae,min_mhc_tcr_pae,min_tcr_p_pae,min_tcr_mhc_pae,peptide_mean_pLDDT_II,assay_type,triad_in_pdb
str,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],list[str],bool,list[f64],list[list[f64]],list[list[f64]],list[f64],f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,list[list[f64]],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,bool
"""0da5bca165be2277f154c5c6907522…",true,"""GQVELGGGNAVEVCK""","""II""","""alpha""","""human""","""DQA1*03:01""","""EDIVADHVASYGVNLYQSYGPSGQYSHEFD…","""beta""","""human""","""DQB1*03:02""","""RDSPEDFVYQFKGMCYFTNGTERVRLVTRY…","""alpha""","""human""","""EDQVTQSPEALRLQEGESSSLNCSYTVSGL…","""beta""","""human""","""KAGVTQTPRYLIKTRGQQVTLSCSPISGHR…","""VSGLRG""","""LYSAGEE""","""TKKE""","""CAVQAGGNNRLAF""","""SGHRS""","""YFSETQ""","""FSNSR""","""CASSLERDGYTF""","[""217432""]","[""1039466""]",false,"[0.49, 0.77, … 0.74]","[[0.02, 0.54, … 0.42], [0.54, 0.88, … 0.82], … [0.42, 0.82, … 0.88]]","[[0.76, 5.93, … 6.42], [4.31, 0.76, … 1.6], … [4.92, 1.67, … 0.76]]","[0.02, 0.88, … 0.88]",0.09,0.0,0.89,0.9,…,0.01119,0.01119,18.043741,10.503677,7.684814,7.34008,0.005035,0.005035,0.000591,0.000591,22,9,"[[0.0, 0.0, … 0.0], [2.0, 0.0, … 1.0], … [0.0, 0.0, … 0.0]]",50.688812,84.24425,92.056793,87.278235,87.717582,86.347297,90.737963,91.859285,88.266451,88.648896,86.432928,18.258717,11.840204,9.80411,14.252087,0.058557,0.058557,6.42,1.35,4.92,1.41,54.996091,"""x-ray crystallography""",true
"""0eab4f1d031514c4d4749bc1a4f6e1…",true,"""RFYKTLRAEQASQ""","""II""","""alpha""","""human""","""DRA*01:02""","""KEEHVIIQAEFYLNPDQSGEFMFDFDGDEI…","""beta""","""human""","""DRB1*11:01""","""GDTRPRFLEYSTSECHFFNGTERVRFLDRY…","""alpha""","""human""","""ILNVEQSPQSLHVQEGDSTNFTCSFPSSNF…","""beta""","""human""","""EPEVTQTPSHQVTQMGQEVILRCVPISNHL…","""SSNFYA""","""MTLNGDE""","""NTKEGY""","""CAFKAAGNKLTF""","""SNHLY""","""FYNNEI""","""PDGSN""","""CASSRLAGGMDEQF""","[""217332""]","[""1033336""]",false,"[0.87, 0.88, … 0.85]","[[0.03, 0.93, … 0.82], [0.93, 0.87, … 0.87], … [0.82, 0.87, … 0.88]]","[[0.76, 0.92, … 1.32], [0.97, 0.76, … 1.15], … [1.26, 1.26, … 0.76]]","[0.03, 0.87, … 0.88]",0.06,0.0,0.9,0.91,…,0.012131,0.012131,5.432119,4.515217,7.488157,7.381104,0.005186,0.005186,0.000468,0.000468,26,15,"[[0.0, 0.0, … 0.0], [2.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]",90.771327,92.430208,90.419423,90.747347,93.5475,94.017045,93.385893,91.437273,91.539801,92.244745,92.098887,5.917691,5.710111,1.896931,3.445875,0.055486,0.055486,1.32,1.15,1.26,1.26,92.937424,"""x-ray crystallography""",true
"""101037d801a41030afabae721c77ec…",true,"""APSGEGSFQPSQENPQGS""","""II""","""alpha""","""human""","""DQA1*03:01""","""EDIVADHVASYGVNLYQSYGPSGQYSHEFD…","""beta""","""human""","""DQB1*03:02""","""RDSPEDFVYQFKGMCYFTNGTERVRLVTRY…","""alpha""","

In [ ]:
pl.read_parquet("../../data/iedb_II/triad/iedb_II_triad.conf_af3.parquet").filter(
    pl.col("cognate")
).explode("receptor_id").join(assay_type, on="receptor_id").filter(
    pl.col("assay_type") == "x-ray crystallography"
).select(
    "job_name"
).unique()

job_name
str
"""cf121d1f52814846fc07a64d294260…"
"""061c8ad983dbd76ee0bfd8065e83e1…"
"""7e8af51467a1660bde66e8b51a47e1…"
"""8820f354b4abe3df9acbd96902f5cf…"
"""f93d2331438af4f4356d1cf38eac85…"
…
"""89095d1cb5f93fbbbf73eddd813e56…"
"""c7954511abf812e94c78d48d9ec1b8…"
"""8071a1d24c713564eb8f87a13dff8d…"


In [ ]:
pl.read_parquet("../../data/iedb_II/triad/iedb_II_triad.conf_af3.parquet").filter(
    pl.col("cognate")
).explode("receptor_id").explode("references").join(
    receptor_reference,
    left_on=["references", "receptor_id"],
    right_on=["reference_id", "receptor_id"],
).filter(
    pl.col("assay_type") == "x-ray crystallography"
).select(
    "job_name"
).unique()

job_name
str
"""f777496168164c075dc2c76ae6358e…"
"""fb295a801f69b721f290dc9413568d…"
"""7e8af51467a1660bde66e8b51a47e1…"
"""cedf34bd3c1ad711c9e71d3018068e…"
"""f2d61d58045c3213c5c6640ac60bd4…"
…
"""a8f0fb7e6177b58a8c9577ad1fce8b…"
"""c7954511abf812e94c78d48d9ec1b8…"
"""1a6333fa4bcaee25755d392c7a9b00…"


In [ ]:
iedb_II.filter(pl.col("assay_type") == "x-ray crystallography").select(
    "job_name"
).unique()

job_name
str
"""0da5bca165be2277f154c5c6907522…"
"""40a5bc0bd375733d6c98789dbc83e3…"
"""b2677109639e627973a47b2d06a7fd…"
"""8994534501247aa06ce81d427997be…"
"""20f085497475c83d78c84108ddecee…"
…
"""0eab4f1d031514c4d4749bc1a4f6e1…"
"""8071a1d24c713564eb8f87a13dff8d…"
"""1cdffdbd2b5355468b39c60626bfd6…"


In [19]:
iedb_II_pdb_triad.filter(pl.col("assay_type") == "x-ray crystallography").filter(
    ~pl.col("triad_in_pdb")
)

job_name,cognate,peptide,mhc_class,mhc_1_chain,mhc_1_species,mhc_1_name,mhc_1_seq,mhc_2_chain,mhc_2_species,mhc_2_name,mhc_2_seq,tcr_1_chain,tcr_1_species,tcr_1_seq,tcr_2_chain,tcr_2_species,tcr_2_seq,tcr_1_cdr_1,tcr_1_cdr_2,tcr_1_cdr_2_5,tcr_1_cdr_3,tcr_2_cdr_1,tcr_2_cdr_2,tcr_2_cdr_2_5,tcr_2_cdr_3,receptor_id,references,pmhc_in_pdb,pmhc_matches,peptide_alignments,triad_in_pdb,triad_matches,assay_type
str,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],bool,list[str],list[str],bool,list[str],str
"""3bf8262290d0427d7005c292d6d326…",true,"""QRCRVHFMRNLYTAV""","""II""","""alpha""","""human""","""DRA*01:02""","""KEEHVIIQAEFYLNPDQSGEFMFDFDGDEI…","""beta""","""human""","""DRB1*15:01""","""GDTRPRFLWQPKRECHFFNGTERVRFLDRY…","""alpha""","""human""","""SQQGEEDPQALSIQEGENATMNCSYKTSIN…","""beta""","""human""","""GAVVSQHPSWVICKSGTSVKIECRSLDFQA…","""TSINN""","""IRSNERE""","""DTSKKS""","""CATDTTSGTYKYIF""","""DFQATT""","""SNEGSKA""","""ASLTL""","""CSARDLTSGANNEQF""","""158""","[""1014229""]",false,null,null,false,null,"""x-ray crystallography"""
"""71cfe29319f368c4def160dae775c0…",true,"""ALAVLHFYPDKGAKN""","""II""","""alpha""","""human""","""DRA*01:02""","""KEEHVIIQAEFYLNPDQSGEFMFDFDGDEI…","""beta""","""human""","""DRB1*15:01""","""GDTRPRFLWQPKRECHFFNGTERVRFLDRY…","""alpha""","""human""","""SQQGEEDPQALSIQEGENATMNCSYKTSIN…","""beta""","""human""","""GAVVSQHPSWVICKSGTSVKIECRSLDFQA…","""TSINN""","""IRSNERE""","""DTSKKS""","""CATDTTSGTYKYIF""","""DFQATT""","""SNEGSKA""","""ASLTL""","""CSARDLTSGANNEQF""","""158""","[""1014229""]",false,null,null,false,null,"""x-ray crystallography"""
"""061c8ad983dbd76ee0bfd8065e83e1…",true,"""QRCRVHFLRNVLAQV""","""II""","""alpha""","""human""","""DRA*01:02""","""KEEHVIIQAEFYLNPDQSGEFMFDFDGDEI…","""beta""","""human""","""DRB1*15:01""","""GDTRPRFLWQPKRECHFFNGTERVRFLDRY…","""alpha""","""human""","""SQQGEEDPQALSIQEGENATMNCSYKTSIN…","""beta""","""human""","""GAVVSQHPSWVICKSGTSVKIECRSLDFQA…","""TSINN""","""IRSNERE""","""DTSKKS""","""CATDTTSGTYKYIF""","""DFQATT""","""SNEGSKA""","""ASLTL""","""CSARDLTSGANNEQF""","""158""","[""1014229""]",false,null,null,false,null,"""x-ray crystallography"""
"""b8632762b6386b639854d53244ce5c…",true,"""DIALNLPRRI""","""II""","""alpha""","""human""","""DRA*01:02""","""KEEHVIIQAEFYLNPDQSGEFMFDFDGDEI…","""beta""","""human""","""DRB3*03:01""","""GDTRPRFLELLKSECHFFNGTERVRFLERY…","""alpha""","""human""","""AQSVTQPDIHITVSEGASLELRCNYSYGAT…","""beta""","""human""","""DGGITQSPKYLFRKEGQNVTLSCEQNLNHD…","""YGATPY""","""YFSGDTLV""","""KRSQSS""","""CAVGASGNTGKLIF""","""LNHDA""","""SQIVND""","""EKKES""","""CASSLRDGYTGELF""","""515""","[""1025226""]",false,null,null,false,null,"""x-ray crystallography"""
"""ff1914b08deb85107f1ce6206e9e42…",true,"""LIHVNIPKKI""","""II""","""alpha""","""human""","""DRA*01:02""","""KEEHVIIQAEFYLNPDQSGEFMFDFDGDEI…","""beta""","""human""","""DRB3*03:01""","""GDTRPRFLELLKSECHFFNGTERVRFLERY…","""alpha""","""human""","""AQSVTQPDIHITVSEGASLELRCNYSYGAT…","""beta""","""human""","""DGGITQSPKYLFRKEGQNVTLSCEQNLNHD…","""YGATPY""","""YFSGDTLV""","""KRSQSS""","""CAVGASGNTGKLIF""","""LNHDA""","""SQIVND""","""EKKES""","""CASSLRDGYTGELF""","""515""","[""1025226""]",false,null,null,false,null,"""x-ray crystallography"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a184bfb77df61fe6de0205b171bcc8…",true,"""ITREEKPAVTAAPKK""","""II""","""alpha""","""human""","""DRA*01:02""","""KEEHVIIQAEFYLNPDQSGEFMFDFDGDEI…","""beta""","""human""","""DRB1*15:01""","""GDTRPRFLWQPKRECHFFNGTERVRFLDRY…","""alpha""","""human""","""SQQGEEDPQALSIQEGENATMNCSYKTSIN…","""beta""","""human""","""GAVVSQHPSWVICKSGTSVKIECRSLDFQA…","""TSINN""","""IRSNERE""","""DTSKKS""","""CATDTTSGTYKYIF""","""DFQATT""","""SNEGSKA""","""ASLTL""","""CSARDLTSGANNEQF""","""1620""","[""421""]",false,null,null,false,null,"""x-ray crystallography"""
"""171154955a6d6006f1a39b5446416c…",true,"""TLDISNNRLESLPAH""","""II""","""alpha""","""human""","""DRA*01:02